# tides_spin_kozai — a Kozai cycle with tides and relativity

A planet with a distant stellar companion that periodically drives its orbit to high eccentricity. Uses the ADAPTIVE IAS15 integrator plus two REBOUNDx forces at once. Matching the C bit-for-bit here means both programs chose the identical sequence of thousands of adaptive steps.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd reboundx_rust
cargo build --release --example tides_spin_kozai
cd porttest
../target/release/examples/tides_spin_kozai 1000.0
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "reboundx_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "tides_spin_kozai"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
import struct

def unbits(h):
    """Turn a 16-hex-digit IEEE-754 bit pattern back into a float."""
    return struct.unpack("<d", int(h, 16).to_bytes(8, "little"))[0]

def read_state(path):
    """Read one of the raw-bit state dumps into {label: [floats]}."""
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if not parts:
                continue
            key, rest = parts[0], parts[1:]
            vals = []
            for tok in rest:
                if len(tok) == 16:
                    try:
                        vals.append(unbits(tok))
                        continue
                    except ValueError:
                        pass
                vals.append(tok)
            out.setdefault(key, []).append(vals)
    return out

def compare(a, b, label_a="C", label_b="Rust"):
    """Byte-compare two dump files and report."""
    ta = open(a, "rb").read().replace(b"\r\n", b"\n")
    tb = open(b, "rb").read().replace(b"\r\n", b"\n")
    if ta == tb:
        print(f"BIT-IDENTICAL: {label_a} and {label_b} agree on every bit")
        return True
    print(f"MISMATCH between {label_a} and {label_b}")
    la, lb = ta.decode().splitlines(), tb.decode().splitlines()
    for i, (x, y) in enumerate(zip(la, lb)):
        if x != y:
            print(f"  line {i}:\n    {label_a}: {x}\n    {label_b}: {y}")
    return False


In [3]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe, "1000.0"], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

kozai done t=1.00000000000000000e3



In [4]:
name = EXAMPLE.replace("tides_spin_", "")
rust = os.path.join(WORK, f"state_{name}_rust.txt")
cref = os.path.join(WORK, f"state_{name}_c.txt")
if os.path.exists(rust):
    st = read_state(rust)
    print("--- final state (decoded from raw bits) ---")
    for p in st.get("p", []):
        print(f"  particle {p[0]}: x={p[1]:+.9e} y={p[2]:+.9e} z={p[3]:+.9e}")
    for o in st.get("Omega", []):
        if len(o) > 1 and not isinstance(o[1], str):
            mag = (o[1]**2 + o[2]**2 + o[3]**2) ** 0.5
            print(f"  spin {o[0]}: |Omega| = {mag:.9e}")
if os.path.exists(cref):
    print()
    compare(cref, rust)
else:
    print("\n(no C reference dump present - build and run the C harness to compare)")


--- final state (decoded from raw bits) ---
  particle 0: x=+1.681799844e+01 y=+1.876832500e+01 z=+8.136347679e-04
  particle 1: x=+1.642726991e+01 y=+1.911456567e+01 z=-1.929832814e+00
  particle 2: x=-1.681884560e+01 y=-1.876931074e+01 z=-7.141132897e-04
  spin 0: |Omega| = 7.934782609e+01
  spin 1: |Omega| = 3.650000000e+02

MISMATCH between C and Rust
  line 0:
    C: example tides_spin_kozai tmax 40f86a0000000000
    Rust: example tides_spin_kozai tmax 408f400000000000
  line 1:
    C: t 40f86a0000000000
    Rust: t 408f400000000000
  line 2:
    C: dt 3fdd5b7075dd539c
    Rust: dt 3fdfb0353c9af096
  line 4:
    C: p 0 402b1869e5c02b17 403533d074e7e486 3f4ecd7cc19fdc5d bfb590bd25ad7d0d 3faaf00cb437a37b 3ef18758a3ed87f6 3ff0000000000000
    Rust: p 0 4030d168588d7d87 4032c4b0f270001d 3f4aa9435c08259e bfb30eb4eb3d86b0 3fb0c5d3841351ea bed2f14dd73ad188 3ff0000000000000
  line 5:
    C: Omega 0 3f072bc1efc6c3aa 40538913d4bbd5e5 402b90271149db94
    Rust: Omega 0 3e998f76a30319ba 40538